In [ ]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

df = pd.read_csv("../../data/df_processed.csv",index_col=0)
train_mask = df["MathScore"].notna()

y_train = df.loc[train_mask, "MathScore"]
X = df.drop(columns=["MathScore"])
X_train, X_test = X.loc[train_mask], X.loc[~train_mask]

exo_embedding_finetuned_full_train = pd.read_csv("../../data/Embedding_wandb/exo_embedding_finetuned_full_train.csv",index_col=0)
que_embedding_finetuned_full_train = pd.read_csv("../../data/Embedding_wandb/que_embedding_finetuned_full_train.csv",index_col=0)

X_train = X_train.join(que_embedding_finetuned_full_train, how="left")
X_train = X_train.join(exo_embedding_finetuned_full_train, how="left")
train = X_train.join(y_train, how="left")



train_data = TabularDataset(train)
train_data.head()

In [ ]:
label = 'MathScore'
time_limit = 15*60
metric = 'r2'
predictor = TabularPredictor(label, eval_metric=metric).fit(train_data, time_limit=time_limit, presets='extreme_quality', num_gpus=1)

In [ ]:
exo_embedding_finetuned_full_test = pd.read_csv("../../data/Embedding_wandb/exo_embedding_finetuned_full_test.csv",index_col=0)
que_embedding_finetuned_full_test = pd.read_csv("../../data/Embedding_wandb/que_embedding_finetuned_full_test.csv",index_col=0)
X_test = X_test.join(que_embedding_finetuned_full_train, how="left")
X_test = X_test.join(exo_embedding_finetuned_full_train, how="left")
test_data = TabularDataset(X_test)
test_data.head()
#=======
y_pred = predictor.predict(test_data)
y_pred.head()  # Predictions